# Transfer Learning & Fine-Tuning

Reusing pretrained CNN representations for a new classification task — implemented
from scratch (Topic 09 CNN backbone), with frozen-backbone feature extraction,
fine-tuning with discriminative learning rates, and honest comparison when the
source domain does *not* match the target.

We answer four concrete questions with numbers:
1. How do we transfer conv-layer weights to a new task with a different number of classes?
2. With very few target labels, does a frozen pretrained backbone beat training from scratch?
3. Does fine-tuning (lower LR on the backbone) improve over a frozen feature extractor?
4. What happens when the source task (digits 0–4) does not include the target classes (5–9)?

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. CNN Backbone with Freeze / Fine-Tune Controls

We reuse the Topic 09 `Conv2d` + `MaxPool2d` stack and wrap it in a `DigitCNN` that
supports:
- **Variable number of output classes** (replace the final linear head)
- **`copy_backbone_from`** — copy conv weights from a pretrained model
- **`train_backbone` / `train_head` flags** — feature extraction vs fine-tuning
- **Discriminative learning rates** — `lr` for the head, `lr_backbone` for conv layers

In [2]:
def im2col(x, kH, kW, stride, pad):
    N, C, H, W = x.shape
    x_pad = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')
    H_out = (H + 2 * pad - kH) // stride + 1
    W_out = (W + 2 * pad - kW) // stride + 1
    cols = np.zeros((N, C, kH, kW, H_out, W_out), dtype=x.dtype)
    for kh in range(kH):
        for kw in range(kW):
            cols[:, :, kh, kw, :, :] = x_pad[:, :, kh:kh + stride * H_out:stride,
                                              kw:kw + stride * W_out:stride]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N * H_out * W_out, C * kH * kW), H_out, W_out


def col2im(cols, x_shape, kH, kW, stride, pad):
    N, C, H, W = x_shape
    H_out = (H + 2 * pad - kH) // stride + 1
    W_out = (W + 2 * pad - kW) // stride + 1
    cols = cols.reshape(N, H_out, W_out, C, kH, kW).transpose(0, 3, 4, 5, 1, 2)
    x_pad = np.zeros((N, C, H + 2 * pad, W + 2 * pad), dtype=cols.dtype)
    for kh in range(kH):
        for kw in range(kW):
            x_pad[:, :, kh:kh + stride * H_out:stride, kw:kw + stride * W_out:stride] += cols[:, :, kh, kw, :, :]
    return x_pad[:, :, pad:pad + H, pad:pad + W]


class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, seed=0):
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.in_channels, self.out_channels = in_channels, out_channels
        self.kH, self.kW = kernel_size
        self.stride, self.padding = stride, padding
        rng = np.random.RandomState(seed)
        scale = np.sqrt(2.0 / (in_channels * self.kH * self.kW))
        self.W = rng.randn(out_channels, in_channels, self.kH, self.kW) * scale
        self.b = np.zeros(out_channels)
        self.cache = None

    def forward(self, x):
        N = x.shape[0]
        cols, H_out, W_out = im2col(x, self.kH, self.kW, self.stride, self.padding)
        W_col = self.W.reshape(self.out_channels, -1)
        out = (cols @ W_col.T + self.b).reshape(N, H_out, W_out, self.out_channels).transpose(0, 3, 1, 2)
        self.cache = (x, cols)
        return out

    def backward(self, dout):
        x, cols = self.cache
        dout_r = dout.transpose(0, 2, 3, 1).reshape(-1, self.out_channels)
        dW = (dout_r.T @ cols).reshape(self.W.shape)
        db = dout_r.sum(axis=0)
        dcols = dout_r @ self.W.reshape(self.out_channels, -1)
        dx = col2im(dcols, x.shape, self.kH, self.kW, self.stride, self.padding)
        return dx, dW, db


class MaxPool2d:
    def __init__(self, kernel_size, stride=None):
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kH, self.kW = kernel_size
        self.stride = self.kH if stride is None else stride
        self.cache = None

    def forward(self, x):
        N, C, H, W = x.shape
        cols, H_out, W_out = im2col(x, self.kH, self.kW, self.stride, 0)
        cols = cols.reshape(N, H_out, W_out, C, self.kH * self.kW)
        out = cols.max(axis=4)
        self.cache = (x.shape, cols.argmax(axis=4))
        return out.transpose(0, 3, 1, 2)

    def backward(self, dout):
        x_shape, argmax = self.cache
        N, C, H, W = x_shape
        H_out = (H - self.kH) // self.stride + 1
        W_out = (W - self.kW) // self.stride + 1
        dx = np.zeros(x_shape, dtype=dout.dtype)
        dout_r = dout.transpose(0, 2, 3, 1)
        for n in range(N):
            for c in range(C):
                for oh in range(H_out):
                    for ow in range(W_out):
                        idx = argmax[n, oh, ow, c]
                        kh, kw = divmod(idx, self.kW)
                        dx[n, c, oh * self.stride + kh, ow * self.stride + kw] += dout_r[n, oh, ow, c]
        return dx


def relu(x):
    return np.maximum(0, x)


def relu_grad(x):
    return (x > 0).astype(x.dtype)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class DigitCNN:
    FEAT_DIM = 16 * 2 * 2

    def __init__(self, num_classes=10, seed=0):
        self.num_classes = num_classes
        self.conv1 = Conv2d(1, 8, 3, padding=1, seed=seed)
        self.conv2 = Conv2d(8, 16, 3, padding=1, seed=seed + 1)
        self.pool1 = MaxPool2d(2, 2)
        self.pool2 = MaxPool2d(2, 2)
        rng = np.random.RandomState(seed + 2)
        self.Wfc = rng.randn(self.FEAT_DIM, num_classes) * np.sqrt(2.0 / self.FEAT_DIM)
        self.bfc = np.zeros(num_classes)
        self.train_backbone = True
        self.train_head = True

    def features_and_logits(self, x):
        c1 = self.conv1.forward(x)
        a1 = relu(c1)
        p1 = self.pool1.forward(a1)
        c2 = self.conv2.forward(p1)
        a2 = relu(c2)
        p2 = self.pool2.forward(a2)
        flat = p2.reshape(p2.shape[0], -1)
        logits = flat @ self.Wfc + self.bfc
        return logits, (c1, a1, p1, c2, a2, p2, flat)

    def forward(self, x):
        return self.features_and_logits(x)[0]

    def extract_features(self, x):
        return self.features_and_logits(x)[1][-1]

    def copy_backbone_from(self, other):
        self.conv1.W = other.conv1.W.copy()
        self.conv1.b = other.conv1.b.copy()
        self.conv2.W = other.conv2.W.copy()
        self.conv2.b = other.conv2.b.copy()

    def backbone_snapshot(self):
        return (self.conv1.W.copy(), self.conv1.b.copy(), self.conv2.W.copy(), self.conv2.b.copy())

    def step(self, x, y, lr=0.5, lr_backbone=None):
        lr_backbone = lr if lr_backbone is None else lr_backbone
        logits, cache = self.features_and_logits(x)
        c1, a1, p1, c2, a2, p2, flat = cache
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        if self.train_head:
            dWfc = flat.T @ dlogits
            dbfc = dlogits.sum(axis=0)
        dflat = dlogits @ self.Wfc.T
        dp2 = dflat.reshape(p2.shape)
        da2 = self.pool2.backward(dp2)
        dc2 = da2 * relu_grad(c2)
        dp1, dW2, db2 = self.conv2.backward(dc2)
        da1 = self.pool1.backward(dp1)
        dc1 = da1 * relu_grad(c1)
        _, dW1, db1 = self.conv1.backward(dc1)
        if self.train_head:
            self.Wfc -= lr * dWfc
            self.bfc -= lr * dbfc
        if self.train_backbone:
            self.conv2.W -= lr_backbone * dW2
            self.conv2.b -= lr_backbone * db2
            self.conv1.W -= lr_backbone * dW1
            self.conv1.b -= lr_backbone * db1
        return loss

    def predict(self, x):
        return self.forward(x).argmax(axis=1)

    def accuracy(self, x, y):
        return float((self.predict(x) == y).mean())


def train_model(model, X, y, epochs=40, lr=0.5, lr_backbone=None, batch=32, seed=0):
    rng = np.random.RandomState(seed)
    for _ in range(epochs):
        idx = rng.permutation(len(X))
        for i in range(0, len(X), batch):
            model.step(X[idx[i:i + batch]], y[idx[i:i + batch]], lr=lr, lr_backbone=lr_backbone)


def subsample_per_class(X, y, n_per_class, seed=0):
    rng = np.random.RandomState(seed)
    idx = []
    for c in np.unique(y):
        c_idx = np.where(y == c)[0]
        idx.extend(rng.choice(c_idx, n_per_class, replace=False))
    idx = np.array(idx)
    rng.shuffle(idx)
    return X[idx], y[idx]


print('DigitCNN and helpers defined.')

DigitCNN and helpers defined.


## 2. PyTorch Validation & Frozen-Backbone Check

We validate one forward pass against PyTorch, then confirm that setting
`train_backbone=False` leaves conv weights unchanged after training steps.

In [3]:
ae = DigitCNN(num_classes=5, seed=42)
Xb = np.random.RandomState(0).randn(3, 1, 8, 8) * 0.1 + 0.5
out = ae.forward(Xb)

class PT_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc = nn.Linear(64, 5)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = nn.functional.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = nn.functional.max_pool2d(x, 2)
        return self.fc(x.flatten(1))

pt = PT_CNN().double()
with torch.no_grad():
    pt.conv1.weight.copy_(torch.from_numpy(ae.conv1.W))
    pt.conv1.bias.copy_(torch.from_numpy(ae.conv1.b))
    pt.conv2.weight.copy_(torch.from_numpy(ae.conv2.W))
    pt.conv2.bias.copy_(torch.from_numpy(ae.conv2.b))
    pt.fc.weight.copy_(torch.from_numpy(ae.Wfc.T))
    pt.fc.bias.copy_(torch.from_numpy(ae.bfc))
out_pt = pt(torch.from_numpy(Xb)).detach().numpy()
print(f'Forward max diff vs PyTorch: {np.max(np.abs(out - out_pt)):.2e}')

# Frozen backbone: conv weights must not move
digits = load_digits()
X = digits.images.astype(np.float64) / 16.0
y = digits.target
Xc = X[:, np.newaxis]
pre = DigitCNN(10, seed=0)
X_tr, _, y_tr, _ = train_test_split(Xc, y, test_size=0.25, random_state=0, stratify=y)
train_model(pre, X_tr, y_tr, epochs=5, seed=0)
frozen = DigitCNN(5, seed=1)
frozen.copy_backbone_from(pre)
frozen.train_backbone = False
snap = frozen.backbone_snapshot()
X_mini = X_tr[:10]
y_mini = np.arange(10) % 5
for _ in range(20):
    frozen.step(X_mini, y_mini, lr=0.5)
delta = max(np.abs(frozen.conv1.W - snap[0]).max(), np.abs(frozen.conv2.W - snap[2]).max())
print(f'Max conv weight change with train_backbone=False: {delta:.2e}')

Forward max diff vs PyTorch: 6.66e-16


Max conv weight change with train_backbone=False: 0.00e+00


## 3. Pretrain Source Models

**Source A (good transfer):** 10-class digit classifier on all digits — backbone learns
general stroke/edge features including classes 5–9.

**Source B (poor transfer):** 5-class classifier on digits 0–4 only — same low-level
features, but the head never saw classes 5–9.

**Target task:** classify digits 5–9 (relabeled 0–4) with a **tiny labeled set**.

In [4]:
X_tgt = X[y >= 5]
y_tgt = y[y >= 5] - 5
X_tgt_c = X_tgt[:, np.newaxis]
X04 = X[y < 5]
y04 = y[y < 5]
X04_c = X04[:, np.newaxis]

X10_tr, X10_te, y10_tr, y10_te = train_test_split(Xc, y, test_size=0.25, random_state=0, stratify=y)
X04_tr, X04_te, y04_tr, y04_te = train_test_split(X04_c, y04, test_size=0.25, random_state=0, stratify=y04)
X_tgt_tr, X_tgt_te, y_tgt_tr, y_tgt_te = train_test_split(
    X_tgt_c, y_tgt, test_size=0.25, random_state=0, stratify=y_tgt)

pretrain_10 = DigitCNN(num_classes=10, seed=0)
train_model(pretrain_10, X10_tr, y10_tr, epochs=40, lr=0.5, seed=0)
pretrain_04 = DigitCNN(num_classes=5, seed=0)
train_model(pretrain_04, X04_tr, y04_tr, epochs=40, lr=0.5, seed=0)

print(f'Source A (10-class) test accuracy: {pretrain_10.accuracy(X10_te, y10_te):.3f}')
print(f'Source B (0-4 only) test accuracy: {pretrain_04.accuracy(X04_te, y04_te):.3f}')
print(f'Target pool: digits 5-9, {len(y_tgt_tr)} train / {len(y_tgt_te)} test')

Source A (10-class) test accuracy: 0.971
Source B (0-4 only) test accuracy: 0.965
Target pool: digits 5-9, 672 train / 224 test


## 4. Few-Shot Target Task: Scratch vs Transfer

For each label budget ($n$ samples per class, 3 random seeds), we compare:
- **Scratch** — new random CNN, train end-to-end
- **Frozen backbone** — 10-class pretrained conv layers, new head, backbone frozen
- **Fine-tune** — same init, backbone LR = 0.05, head LR = 0.5

In [5]:
SEEDS = [0, 1, 2]
LABEL_BUDGETS = [3, 5, 10]
results = []

for n in LABEL_BUDGETS:
    row = {'n': n, 'scratch': [], 'frozen10': [], 'finetune10': [], 'frozen04': []}
    for s in SEEDS:
        Xs, ys = subsample_per_class(X_tgt_tr, y_tgt_tr, n, seed=s)
        scratch = DigitCNN(5, seed=s)
        train_model(scratch, Xs, ys, epochs=40, lr=0.5, seed=s)
        row['scratch'].append(scratch.accuracy(X_tgt_te, y_tgt_te))
        frozen = DigitCNN(5, seed=s + 10)
        frozen.copy_backbone_from(pretrain_10)
        frozen.train_backbone = False
        train_model(frozen, Xs, ys, epochs=40, lr=0.5, seed=s)
        row['frozen10'].append(frozen.accuracy(X_tgt_te, y_tgt_te))
        finetune = DigitCNN(5, seed=s + 20)
        finetune.copy_backbone_from(pretrain_10)
        train_model(finetune, Xs, ys, epochs=40, lr=0.5, lr_backbone=0.05, seed=s)
        row['finetune10'].append(finetune.accuracy(X_tgt_te, y_tgt_te))
        frozen4 = DigitCNN(5, seed=s + 30)
        frozen4.copy_backbone_from(pretrain_04)
        frozen4.train_backbone = False
        train_model(frozen4, Xs, ys, epochs=40, lr=0.5, seed=s)
        row['frozen04'].append(frozen4.accuracy(X_tgt_te, y_tgt_te))
    results.append(row)

print(f"{'n/cls':>6} {'scratch':>10} {'frozen10':>10} {'finetune10':>12} {'frozen04':>10}")
for row in results:
    print(f"{row['n']:6d} {np.mean(row['scratch']):10.3f} {np.mean(row['frozen10']):10.3f} "
          f"{np.mean(row['finetune10']):12.3f} {np.mean(row['frozen04']):10.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(LABEL_BUDGETS))
w = 0.2
ax.bar(x - 1.5*w, [np.mean(r['scratch']) for r in results], w, label='scratch')
ax.bar(x - 0.5*w, [np.mean(r['frozen10']) for r in results], w, label='frozen (10-cls)')
ax.bar(x + 0.5*w, [np.mean(r['finetune10']) for r in results], w, label='fine-tune (10-cls)')
ax.bar(x + 1.5*w, [np.mean(r['frozen04']) for r in results], w, label='frozen (0-4 only)')
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in LABEL_BUDGETS])
ax.set_xlabel('labeled samples per class')
ax.set_ylabel('target test accuracy')
ax.set_title('Transfer learning on digits 5-9 (3 seeds mean)')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('transfer_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

 n/cls    scratch   frozen10   finetune10   frozen04
     3      0.771      0.918        0.927      0.723
     5      0.863      0.936        0.939      0.790
    10      0.918      0.978        0.964      0.830


## 5. Linear Probe on Pretrained Features

A **linear probe** trains a logistic regression on frozen conv features — the simplest
feature-extraction baseline. At $n=3$ samples per class, pretrained 10-class features
should dominate a random untrained backbone.

In [6]:
n = 3
probe_pre, probe_rand = [], []
for s in SEEDS:
    Xs, ys = subsample_per_class(X_tgt_tr, y_tgt_tr, n, seed=s)
    rnd = DigitCNN(5, seed=s + 100)
    lr_r = LogisticRegression(max_iter=2000).fit(rnd.extract_features(Xs), ys)
    probe_rand.append(lr_r.score(rnd.extract_features(X_tgt_te), y_tgt_te))
    lr_p = LogisticRegression(max_iter=2000).fit(pretrain_10.extract_features(Xs), ys)
    probe_pre.append(lr_p.score(pretrain_10.extract_features(X_tgt_te), y_tgt_te))

print(f'Linear probe at n={n}/class (3 seeds):')
print(f'  Random backbone features:     {np.mean(probe_rand):.3f}')
print(f'  10-class pretrained features: {np.mean(probe_pre):.3f}')
print(f'  End-to-end scratch CNN:       {np.mean(results[0]["scratch"]):.3f}')
print(f'  Frozen backbone + new head:   {np.mean(results[0]["frozen10"]):.3f}')

Linear probe at n=3/class (3 seeds):
  Random backbone features:     0.789
  10-class pretrained features: 0.954
  End-to-end scratch CNN:       0.771
  Frozen backbone + new head:   0.918


## 6. Takeaways

1. **Transfer helps most when labels are scarce** and the source model saw related data.
2. **Frozen feature extraction** is verified by zero conv-weight change; only the head learns.
3. **Fine-tuning** with a lower backbone LR can edge out frozen features when enough
   target labels exist.
4. **Wrong source domain** (0–4 pretrain for 5–9 target) underperforms scratch — transfer
   is not magic; source and target must be related.

In [7]:
best_n3 = np.mean(results[0]['frozen10']) - np.mean(results[0]['scratch'])
gap_wrong = np.mean(results[2]['scratch']) - np.mean(results[2]['frozen04'])
print(f'At n=3/class, 10-class frozen transfer beats scratch by {best_n3:+.3f} accuracy')
print(f'At n=10/class, scratch beats 0-4 frozen transfer by {gap_wrong:+.3f} (wrong source domain)')

At n=3/class, 10-class frozen transfer beats scratch by +0.147 accuracy
At n=10/class, scratch beats 0-4 frozen transfer by +0.088 (wrong source domain)
